# 01 – Probabilistic Population Disaggregation

Main model: Gaussian aggregate NLL training and posterior fine-cell update.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


# -----------------------------
# Paths
# -----------------------------

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS_FIGURES = ROOT / "reports" / "figures"
REPORTS_TABLES = ROOT / "reports" / "tables"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Grid constants
# -----------------------------

FINE_H = 100
FINE_W = 100
COARSE_H = 10
COARSE_W = 10

AE_DIM = 64
FINE_PER_COARSE_SIDE = FINE_H // COARSE_H
FINE_PER_COARSE = FINE_PER_COARSE_SIDE ** 2

assert FINE_H % COARSE_H == 0
assert FINE_W % COARSE_W == 0


# -----------------------------
# Training constants
# -----------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 16
LR = 1e-3
EPOCHS = 500
WEIGHT_DECAY = 1e-4

# Small numerical stabilizer for variances
EPS = 1e-6

# Optional observation noise variance for coarse ground truth
TAU2 = 0.0

print("Root:", ROOT)
print("Device:", DEVICE)

In [ ]:
ae_path = DATA_RAW / "alphaearth_features.npy"
pop_path = DATA_RAW / "population_coarse.npy"

if ae_path.exists() and pop_path.exists():
    print("Loading real data...")
    ae = np.load(ae_path)
    y_coarse_grid = np.load(pop_path)

else:
    print("Raw files not found. Creating synthetic placeholder data.")

    rng = np.random.default_rng(42)

    # Synthetic AlphaEarth features
    ae = rng.normal(size=(FINE_H, FINE_W, AE_DIM)).astype(np.float32)

    # Synthetic latent fine population signal
    yy, xx = np.meshgrid(
        np.linspace(-1, 1, FINE_H),
        np.linspace(-1, 1, FINE_W),
        indexing="ij",
    )

    latent = (
        0.8 * ae[..., 0]
        - 0.4 * ae[..., 1]
        + 0.3 * ae[..., 2]
        + 1.0 * np.exp(-4 * (xx**2 + yy**2))
        + 0.2 * rng.normal(size=(FINE_H, FINE_W))
    )

    # Z-score fine latent population
    latent = (latent - latent.mean()) / latent.std()

    # Aggregate to coarse grid
    y_coarse_grid = latent.reshape(
        COARSE_H,
        FINE_PER_COARSE_SIDE,
        COARSE_W,
        FINE_PER_COARSE_SIDE,
    ).sum(axis=(1, 3))

    ae = ae.astype(np.float32)
    y_coarse_grid = y_coarse_grid.astype(np.float32)

print("AE shape:", ae.shape)
print("Coarse population shape:", y_coarse_grid.shape)

assert ae.shape == (FINE_H, FINE_W, AE_DIM)
assert y_coarse_grid.shape == (COARSE_H, COARSE_W)

In [ ]:
# Flatten fine cells
x_fine = ae.reshape(-1, AE_DIM).astype(np.float32)

# Flatten coarse targets
y_coarse = y_coarse_grid.reshape(-1).astype(np.float32)

# Build block id for each fine cell
block_ids_grid = np.zeros((FINE_H, FINE_W), dtype=np.int64)

for r in range(FINE_H):
    for c in range(FINE_W):
        coarse_r = r // FINE_PER_COARSE_SIDE
        coarse_c = c // FINE_PER_COARSE_SIDE
        block_ids_grid[r, c] = coarse_r * COARSE_W + coarse_c

block_ids = block_ids_grid.reshape(-1)

print("x_fine:", x_fine.shape)
print("y_coarse:", y_coarse.shape)
print("block_ids:", block_ids.shape)
print("Number of coarse blocks:", y_coarse.shape[0])
print("Fine cells per coarse block:", np.bincount(block_ids)[:5])

In [ ]:
all_blocks = np.arange(COARSE_H * COARSE_W)

train_blocks, test_blocks = train_test_split(
    all_blocks,
    test_size=0.2,
    random_state=42,
)

train_blocks, val_blocks = train_test_split(
    train_blocks,
    test_size=0.2,
    random_state=42,
)

print("Train coarse cells:", len(train_blocks))
print("Val coarse cells:", len(val_blocks))
print("Test coarse cells:", len(test_blocks))


class CoarseBlockDataset(Dataset):
    def __init__(self, x_fine, y_coarse, block_ids, block_list):
        self.x_fine = torch.tensor(x_fine, dtype=torch.float32)
        self.y_coarse = torch.tensor(y_coarse, dtype=torch.float32)
        self.block_ids = torch.tensor(block_ids, dtype=torch.long)
        self.block_list = torch.tensor(block_list, dtype=torch.long)

        self.indices_by_block = []
        for b in self.block_list:
            idx = torch.where(self.block_ids == b)[0]
            self.indices_by_block.append(idx)

    def __len__(self):
        return len(self.block_list)

    def __getitem__(self, idx):
        block_id = self.block_list[idx]
        fine_idx = self.indices_by_block[idx]

        x_block = self.x_fine[fine_idx]
        y_block = self.y_coarse[block_id]

        return x_block, y_block, block_id


def collate_blocks(batch):
    x_blocks, y_blocks, block_ids_batch = zip(*batch)

    x_blocks = torch.stack(x_blocks, dim=0)
    y_blocks = torch.stack(y_blocks, dim=0)
    block_ids_batch = torch.stack(block_ids_batch, dim=0)

    return x_blocks, y_blocks, block_ids_batch


train_ds = CoarseBlockDataset(x_fine, y_coarse, block_ids, train_blocks)
val_ds = CoarseBlockDataset(x_fine, y_coarse, block_ids, val_blocks)
test_ds = CoarseBlockDataset(x_fine, y_coarse, block_ids, test_blocks)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_blocks,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_blocks,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_blocks,
)

In [ ]:
class FineCellGaussianNN(nn.Module):
    def __init__(self, in_dim=64, hidden_dim=128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        """
        x: [..., AE_DIM]

        returns:
            mu: [...], predicted fine-cell mean
            var: [...], predicted fine-cell variance
        """
        out = self.net(x)

        mu = out[..., 0]

        # softplus guarantees positive variance
        raw_var = out[..., 1]
        var = torch.nn.functional.softplus(raw_var) + EPS

        return mu, var


model = FineCellGaussianNN(in_dim=AE_DIM, hidden_dim=128).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

model

In [ ]:
def aggregate_gaussian_nll(mu_fine, var_fine, y_coarse, tau2=0.0):
    """
    mu_fine: [batch, fine_per_coarse]
    var_fine: [batch, fine_per_coarse]
    y_coarse: [batch]

    Returns average aggregate Gaussian NLL.
    """
    mean_agg = mu_fine.sum(dim=1)
    var_agg = var_fine.sum(dim=1) + tau2
    var_agg = torch.clamp(var_agg, min=EPS)

    nll = 0.5 * (
        ((y_coarse - mean_agg) ** 2) / var_agg
        + torch.log(var_agg)
        + np.log(2 * np.pi)
    )

    return nll.mean()


@torch.no_grad()
def evaluate_nll(model, loader, tau2=0.0):
    model.eval()

    losses = []

    for x_block, y_block, _ in loader:
        x_block = x_block.to(DEVICE)
        y_block = y_block.to(DEVICE)

        mu, var = model(x_block)
        loss = aggregate_gaussian_nll(mu, var, y_block, tau2=tau2)

        losses.append(loss.item())

    return float(np.mean(losses))

In [ ]:
history = []

best_val = float("inf")
best_state = None

for epoch in tqdm(range(1, EPOCHS + 1)):
    model.train()

    train_losses = []

    for x_block, y_block, _ in train_loader:
        x_block = x_block.to(DEVICE)
        y_block = y_block.to(DEVICE)

        optimizer.zero_grad()

        mu, var = model(x_block)
        loss = aggregate_gaussian_nll(mu, var, y_block, tau2=TAU2)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_nll = float(np.mean(train_losses))
    val_nll = evaluate_nll(model, val_loader, tau2=TAU2)

    history.append(
        {
            "epoch": epoch,
            "train_nll": train_nll,
            "val_nll": val_nll,
        }
    )

    if val_nll < best_val:
        best_val = val_nll
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

    if epoch % 50 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:04d} | "
            f"train NLL = {train_nll:.4f} | "
            f"val NLL = {val_nll:.4f}"
        )

model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.tail()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_nll"], label="train")
plt.plot(history_df["epoch"], history_df["val_nll"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Aggregate Gaussian NLL")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict_aggregate_distribution(model, loader, tau2=0.0):
    model.eval()

    rows = []

    for x_block, y_block, block_id in loader:
        x_block = x_block.to(DEVICE)

        mu, var = model(x_block)

        mean_agg = mu.sum(dim=1).cpu()
        var_agg = (var.sum(dim=1) + tau2).cpu()
        std_agg = torch.sqrt(var_agg)

        for b, y, m, v, s in zip(
            block_id.cpu().numpy(),
            y_block.cpu().numpy(),
            mean_agg.numpy(),
            var_agg.numpy(),
            std_agg.numpy(),
        ):
            rows.append(
                {
                    "block_id": int(b),
                    "y_true": float(y),
                    "mean_pred": float(m),
                    "var_pred": float(v),
                    "std_pred": float(s),
                }
            )

    return pd.DataFrame(rows)


test_pred = predict_aggregate_distribution(model, test_loader, tau2=TAU2)
test_pred.head()

In [ ]:
def add_interval_coverage(df, levels=(0.5, 0.8, 0.9, 0.95)):
    out = []

    mean = torch.tensor(df["mean_pred"].values, dtype=torch.float32)
    std = torch.tensor(df["std_pred"].values, dtype=torch.float32)
    y = torch.tensor(df["y_true"].values, dtype=torch.float32)

    dist = Normal(mean, std)

    for level in levels:
        alpha = 1.0 - level

        lower = dist.icdf(torch.tensor(alpha / 2))
        upper = dist.icdf(torch.tensor(1.0 - alpha / 2))

        inside = ((y >= lower) & (y <= upper)).float()
        width = upper - lower

        out.append(
            {
                "nominal_coverage": level,
                "empirical_coverage": inside.mean().item(),
                "avg_interval_width": width.mean().item(),
            }
        )

    return pd.DataFrame(out)


coverage_df = add_interval_coverage(test_pred)
coverage_df

In [ ]:
plt.figure(figsize=(5, 5))
plt.plot(
    coverage_df["nominal_coverage"],
    coverage_df["empirical_coverage"],
    marker="o",
    label="model",
)
plt.plot([0, 1], [0, 1], linestyle="--", label="ideal")
plt.xlabel("Nominal coverage")
plt.ylabel("Empirical coverage")
plt.title("Predictive interval calibration")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
mean = torch.tensor(test_pred["mean_pred"].values, dtype=torch.float32)
std = torch.tensor(test_pred["std_pred"].values, dtype=torch.float32)
y = torch.tensor(test_pred["y_true"].values, dtype=torch.float32)

dist = Normal(mean, std)

pit = dist.cdf(y).numpy()
z = ((y - mean) / std).numpy()

print("Mean standardized residual:", z.mean())
print("Std standardized residual:", z.std())

plt.figure(figsize=(6, 4))
plt.hist(pit, bins=10, edgecolor="black")
plt.xlabel("PIT value")
plt.ylabel("Count")
plt.title("PIT histogram")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(z, bins=10, edgecolor="black")
plt.xlabel("Standardized residual")
plt.ylabel("Count")
plt.title("Standardized residuals")
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict_fine_prior(model, x_fine):
    model.eval()

    x = torch.tensor(x_fine, dtype=torch.float32, device=DEVICE)

    mu_list = []
    var_list = []

    batch_size = 2048

    for start in range(0, len(x), batch_size):
        xb = x[start:start + batch_size]
        mu, var = model(xb)

        mu_list.append(mu.cpu())
        var_list.append(var.cpu())

    mu_fine = torch.cat(mu_list).numpy()
    var_fine = torch.cat(var_list).numpy()

    return mu_fine, var_fine


mu_prior, var_prior = predict_fine_prior(model, x_fine)

mu_post = np.zeros_like(mu_prior)
var_post = np.zeros_like(var_prior)

for b in range(COARSE_H * COARSE_W):
    idx = np.where(block_ids == b)[0]

    mu_b = mu_prior[idx]
    var_b = var_prior[idx]

    y_b = y_coarse[b]

    mean_agg = mu_b.sum()
    var_agg = var_b.sum() + TAU2

    residual = y_b - mean_agg

    # Gaussian conditioning update
    mu_post[idx] = mu_b + (var_b / var_agg) * residual
    var_post[idx] = var_b - (var_b ** 2) / var_agg

# Reshape to fine maps
mu_prior_grid = mu_prior.reshape(FINE_H, FINE_W)
std_prior_grid = np.sqrt(var_prior.reshape(FINE_H, FINE_W))

mu_post_grid = mu_post.reshape(FINE_H, FINE_W)
std_post_grid = np.sqrt(np.maximum(var_post.reshape(FINE_H, FINE_W), 0.0))

print("Prior mean grid:", mu_prior_grid.shape)
print("Posterior mean grid:", mu_post_grid.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(mu_post_grid)
plt.colorbar(label="Posterior mean")
plt.title("Fine-grid posterior mean")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(std_post_grid)
plt.colorbar(label="Posterior std")
plt.title("Fine-grid posterior uncertainty")
plt.tight_layout()
plt.show()

In [ ]:
posterior_agg = np.zeros(COARSE_H * COARSE_W)

for b in range(COARSE_H * COARSE_W):
    idx = np.where(block_ids == b)[0]
    posterior_agg[b] = mu_post[idx].sum()

check_df = pd.DataFrame(
    {
        "block_id": np.arange(COARSE_H * COARSE_W),
        "observed_coarse": y_coarse,
        "posterior_mean_sum": posterior_agg,
        "error": posterior_agg - y_coarse,
    }
)

check_df.head()

In [ ]:
print("Mean absolute aggregate error:", np.abs(check_df["error"]).mean())
print("Max absolute aggregate error:", np.abs(check_df["error"]).max())

plt.figure(figsize=(5, 5))
plt.scatter(check_df["observed_coarse"], check_df["posterior_mean_sum"])
plt.xlabel("Observed coarse value")
plt.ylabel("Posterior fine means aggregated")
plt.title("Posterior aggregate consistency")
plt.tight_layout()
plt.show()

In [ ]:
np.save(DATA_PROCESSED / "mu_prior_grid.npy", mu_prior_grid)
np.save(DATA_PROCESSED / "std_prior_grid.npy", std_prior_grid)
np.save(DATA_PROCESSED / "mu_post_grid.npy", mu_post_grid)
np.save(DATA_PROCESSED / "std_post_grid.npy", std_post_grid)

coverage_df.to_csv(REPORTS_TABLES / "coverage_metrics.csv", index=False)
test_pred.to_csv(REPORTS_TABLES / "test_aggregate_predictions.csv", index=False)

print("Saved posterior maps and evaluation tables.")